In [1]:
import re
import json
import pandas as pd

def extract_route_data(html_content):
    # Extract rawShapes JSON
    raw_match = re.search(
        r'const rawShapes = \$\.parseJSON\(`(.*?)`\);',
        html_content,
        re.S
    )

    if not raw_match:
        raise ValueError("rawShapes not found")

    raw_shapes = json.loads(raw_match.group(1))

    # Extract bus stops JSON
    stops_match = re.search(
        r'var bstp = (\[.*?\]);',
        html_content,
        re.S
    )

    if not stops_match:
        raise ValueError("bstp (bus stops) not found")

    stops = json.loads(stops_match.group(1))

    return raw_shapes, stops


# ===== LOAD SAVED HTML FILE =====
with open("C:/Users/C00513/Documents/TM/SMART FMS/FastAPI_Exercise/Initiative_2.0/RouteA.html", "r", encoding="utf-8") as f:
    html = f.read()

raw_shapes, stops = extract_route_data(html)

print("Directions found:", raw_shapes.keys())
print("Total stops:", len(stops))
print("First route coordinate:", raw_shapes[list(raw_shapes.keys())[0]][0])
print("First stop:", stops[0])

# ===== CONVERT TO DATAFRAMES =====
# 1️⃣ BUS STOPS TO DATAFRAME
df_stops = pd.DataFrame(stops)

# Optional: reorder columns
df_stops = df_stops[
    ["stop_id", "stop_name", "street_name", "lat", "lng", "dr", "zone"]
]

# 2️⃣ ROUTE COORDINATES TO DATAFRAME
direction_key = list(raw_shapes.keys())[0]  # '02'

route_rows = []

for idx, (lat, lng) in enumerate(raw_shapes[direction_key]):
    route_rows.append({
        "sequence": idx,
        "direction": direction_key,
        "latitude": lat,
        "longitude": lng
    })

df_route = pd.DataFrame(route_rows)

# 3️⃣ SAVE TO EXCEL
output_path = "RouteA.xlsx"

with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
    df_stops.to_excel(writer, sheet_name="Bus_Stops", index=False)
    df_route.to_excel(writer, sheet_name="Route_Coordinates", index=False)

print(f"Excel saved to {output_path}")

Directions found: dict_keys(['02'])
Total stops: 89
First route coordinate: [3.054958299344183, 101.7433072149168]
First stop: {'stop_id': '1008352', 'stop_name': 'KL423 PRIMA ALAM DAMAI', 'street_name': 'JLN DESA CHERAS', 'lat': 3.055034, 'lng': 101.743311, 'dr': '1', 'zone': '1'}
Excel saved to RouteA.xlsx


Generate random value to total passenger

In [6]:
import pandas as pd
import numpy as np

df = pd.read_excel("RouteA.xlsx", sheet_name="Route_Coordinates")

df['Total Passenger'] = np.random.randint(1, 40, df.shape[0])

df['Route ID'] = 400

df['Bus ID'] = 'A1'

df['Datetime'] = pd.to_datetime('2024-06-01 08:00:00') + pd.to_timedelta(df['sequence'], unit='m')
 
df.to_csv("routeABaru.csv", index=False)

In [1]:
import re
import json
import pandas as pd
import numpy as np


def extract_route_data(html_content):
    # Extract rawShapes JSON
    raw_match = re.search(
        r'const rawShapes = \$\.parseJSON\(`(.*?)`\);',
        html_content,
        re.S
    )

    if not raw_match:
        raise ValueError("rawShapes not found")

    raw_shapes = json.loads(raw_match.group(1))

    # Extract bus stops JSON
    stops_match = re.search(
        r'var bstp = (\[.*?\]);',
        html_content,
        re.S
    )

    if not stops_match:
        raise ValueError("bstp (bus stops) not found")

    stops = json.loads(stops_match.group(1))

    return raw_shapes, stops


# ===== LOAD HTML FILE =====
html_path = r"C:/Users/C00513/Documents/TM/SMART FMS/FastAPI_Exercise/Initiative_2.0/RouteA.html"

with open(html_path, "r", encoding="utf-8") as f:
    html = f.read()

raw_shapes, stops = extract_route_data(html)

print("Directions found:", raw_shapes.keys())
print("Total stops:", len(stops))


# =========================
# 1️⃣ BUS STOPS DATAFRAME
# =========================
df_stops = pd.DataFrame(stops)

df_stops = df_stops[
    ["stop_id", "stop_name", "street_name", "lat", "lng", "dr", "zone"]
]


# =========================
# 2️⃣ ROUTE COORDINATES
# =========================
direction_key = list(raw_shapes.keys())[0]

route_rows = []

for idx, (lat, lng) in enumerate(raw_shapes[direction_key]):
    route_rows.append({
        "sequence": idx,
        "direction": direction_key,
        "latitude": lat,
        "longitude": lng
    })

df_route = pd.DataFrame(route_rows)


# =========================
# 3️⃣ ADD SIMULATION DATA
# =========================
df_route["Total Passenger"] = np.random.randint(1, 40, df_route.shape[0])
df_route["Route ID"] = 400
df_route["Bus ID"] = "A1"

df_route["Datetime"] = (
    pd.to_datetime("2024-06-01 08:00:00")
    + pd.to_timedelta(df_route["sequence"], unit="m")
)


# =========================
# 4️⃣ SAVE EXCEL ONCE
# =========================
output_path = "RouteA_Final.xlsx"

with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
    df_stops.to_excel(writer, sheet_name="Bus_Stops", index=False)
    df_route.to_excel(writer, sheet_name="Route_Coordinates", index=False)

print(f"Excel saved to {output_path}")

Directions found: dict_keys(['02'])
Total stops: 89
Excel saved to RouteA_Final.xlsx


In [5]:
import re
import json
import pandas as pd
import numpy as np


def extract_route_data(html_content):
    raw_match = re.search(
        r'const rawShapes = \$\.parseJSON\(`(.*?)`\);',
        html_content,
        re.S
    )

    if not raw_match:
        raise ValueError("rawShapes not found")

    raw_shapes = json.loads(raw_match.group(1))

    stops_match = re.search(
        r'var bstp = (\[.*?\]);',
        html_content,
        re.S
    )

    if not stops_match:
        raise ValueError("bstp not found")

    stops = json.loads(stops_match.group(1))

    return raw_shapes, stops


# =========================
# 1️⃣ LOAD EXISTING EXCEL
# =========================
excel_path = "RouteA_Final.xlsx"

df_stops_existing = pd.read_excel(excel_path, sheet_name="Bus_Stops")
df_route_existing = pd.read_excel(excel_path, sheet_name="Route_Coordinates")


# =========================
# 2️⃣ LOAD SECOND HTML FILE
# =========================
html_path = r"C:\Users\C00513\Documents\TM\SMART FMS\FastAPI_Exercise\Initiative_2.0\RouteK.html"

with open(html_path, "r", encoding="utf-8") as f:
    html = f.read()

raw_shapes, stops = extract_route_data(html)


# =========================
# 3️⃣ CREATE NEW DATAFRAMES
# =========================

# Bus stops
df_stops_new = pd.DataFrame(stops)
df_stops_new = df_stops_new[
    ["stop_id", "stop_name", "street_name", "lat", "lng", "dr", "zone"]
]


# Route coordinates
direction_key = list(raw_shapes.keys())[0]

route_rows = []

for idx, (lat, lng) in enumerate(raw_shapes[direction_key]):
    route_rows.append({
        "sequence": idx,
        "direction": direction_key,
        "latitude": lat,
        "longitude": lng
    })

df_route_new = pd.DataFrame(route_rows)


# =========================
# 4️⃣ ADD SIMULATION DATA
# =========================
df_route_new["Total Passenger"] = np.random.randint(1, 40, df_route_new.shape[0])
df_route_new["Route ID"] = 'T300'
df_route_new["Bus ID"] = "K1"

df_route_new["Datetime"] = (
    pd.to_datetime("2024-06-01 09:00:00")
    + pd.to_timedelta(df_route_new["sequence"], unit="m")
)


# =========================
# 5️⃣ APPEND DATA
# =========================
df_stops_final = pd.concat([df_stops_existing, df_stops_new], ignore_index=True)
df_route_final = pd.concat([df_route_existing, df_route_new], ignore_index=True)


# =========================
# 6️⃣ SAVE BACK TO EXCEL
# =========================
with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
    df_stops_final.to_excel(writer, sheet_name="Bus_Stops", index=False)
    df_route_final.to_excel(writer, sheet_name="Route_Coordinates", index=False)

print("New route appended successfully")

New route appended successfully


Create reverse route for new bus

In [27]:
import pandas as pd
excel_path = "RouteA_Final.xlsx"
df = pd.read_excel(excel_path, sheet_name="Route_Coordinates")
df2 = pd.read_excel(excel_path, sheet_name="Bus_Stops")

In [25]:
# forward buses
df_forward = df.copy()

# reverse buses
df_reverse = df.copy().sort_values("sequence", ascending=False).reset_index(drop=True)

# IMPORTANT: keep original sequence and datetime
df_reverse["sequence"] = df_forward["sequence"].values
df_reverse["Datetime"] = df_forward["Datetime"].values

# change bus id (A1 -> A2)
df_reverse["Bus ID"] = df_reverse["Bus ID"].str[:-1] + "2"

# combine
df_all = pd.concat([df_forward, df_reverse], ignore_index=True)

In [28]:
# forward buses
df_forward = df.copy()

# reverse buses
df_reverse = df.copy().sort_values("sequence", ascending=False).reset_index(drop=True)

# reset sequence so it runs 1 -> N again
df_reverse["sequence"] = range(1, len(df_reverse) + 1)

# change bus id (A1 -> A2)
df_reverse["Bus ID"] = df_reverse["Bus ID"].str[:-1] + "2"

# combine both buses
df_all = pd.concat([df_forward, df_reverse], ignore_index=True)

In [29]:
with pd.ExcelWriter("RouteA_Final.xlsx", engine="openpyxl") as writer:
    df_all.to_excel(writer, sheet_name="Route_Coordinates", index=False)
    df2.to_excel(writer, sheet_name="Bus_Stops", index=False)

In [1]:
import pandas as pd

excel_path = "RouteA_Final.xlsx"

df = pd.read_excel(excel_path, sheet_name="Route_Coordinates")
df2 = pd.read_excel(excel_path, sheet_name="Bus_Stops")

# forward buses (original)
df_forward = df.copy()

reverse_list = []

for bus_id, group in df.groupby("Bus ID"):

    # reverse the route
    rev = group.sort_values("sequence", ascending=False).reset_index(drop=True)

    # reset sequence 0 -> N
    rev["sequence"] = range(len(rev))

    # change bus id A1 -> A2
    rev["Bus ID"] = bus_id[:-1] + "2"

    reverse_list.append(rev)

# combine all reverse routes
df_reverse = pd.concat(reverse_list, ignore_index=True)

# combine forward + reverse
df_all = pd.concat([df_forward, df_reverse], ignore_index=True)

# save back to excel
with pd.ExcelWriter("RouteA_Final.xlsx", engine="openpyxl") as writer:
    df_all.to_excel(writer, sheet_name="Route_Coordinates", index=False)
    df2.to_excel(writer, sheet_name="Bus_Stops", index=False)

In [ ]:
import pandas as pd

excel_path = "RouteA_Final.xlsx"

df = pd.read_excel(excel_path, sheet_name="Route_Coordinates")
df2 = pd.read_excel(excel_path, sheet_name="Bus_Stops")

# make sure datetime is datetime
df["Datetime"] = pd.to_datetime(df["Datetime"])

# rows after 12PM
mask = df["Datetime"].dt.hour >= 12

# change bus ids
df.loc[mask & df["Bus ID"].str.endswith("1"), "Bus ID"] = df["Bus ID"].str[:-1] + "3"
df.loc[mask & df["Bus ID"].str.endswith("2"), "Bus ID"] = df["Bus ID"].str[:-1] + "4"

# save
with pd.ExcelWriter("RouteA_Final.xlsx", engine="openpyxl") as writer:
    df.to_excel(writer, sheet_name="Route_Coordinates", index=False)
    df2.to_excel(writer, sheet_name="Bus_Stops", index=False)

: 

Create database for storing bus stop station

In [1]:
import sqlite3

# filename to form database
file = "rapidkl.db"

try:
  conn = sqlite3.connect(file)
  print("Database rapidkl.db formed.")
except:
  print("Database Sqlite3.db not formed.")

Database rapidkl.db formed.


In [2]:
# Importing Sqlite3 Module
import sqlite3

try:
    # Making a connection between sqlite3 database and Python Program
    sqliteConnection = sqlite3.connect('rapidkl.db')
    # If sqlite3 makes a connection with python program then it will print "Connected to SQLite"
    # Otherwise it will show errors
    print("Connected to SQLite")
except sqlite3.Error as error:
    print("Failed to connect with sqlite3 database", error)
finally:
    # Inside Finally Block, If connection is open, we need to close it
    if sqliteConnection:
        # using close() method, we will close the connection
        sqliteConnection.close()
        # After closing connection object, we will print "the sqlite connection is closed"
        print("the sqlite connection is closed")

Connected to SQLite
the sqlite connection is closed


In [ ]:
import sqlite3

# Connect to the SQLite database (or create it if it doesn't exist)
connection_obj = sqlite3.connect('rapidkl.db')

# Create a cursor object to interact with the database
cursor_obj = connection_obj.cursor()

# Drop the GEEK table if it already exists (for clean setup)
cursor_obj.execute("DROP TABLE IF EXISTS GEEK")

# SQL query to create the table
table_creation_query = """
    CREATE TABLE  (
        Email VARCHAR(255) NOT NULL,
        First_Name CHAR(25) NOT NULL,
        Last_Name CHAR(25),
        Score INT
    );
"""

# Execute the table creation query
cursor_obj.execute(table_creation_query)

# Confirm that the table has been created
print("Table is Ready")

# Close the connection to the database
connection_obj.close()